# Lab 08 - Retrieval-Augmented Generation (Co. app v2)

**Chapter:** Huyen, *AI Engineering*, Chapter 6 "RAG and Agents", the RAG half: RAG Architecture, Retrieval Algorithms, Retrieval Optimization.
**Environment:** Cluster notebook (your Docker JupyterLab on the KU VPN).
**Time:** about 4-5 hours.

In Lab 7 you hand-picked the documents for every question. Real users don't come with hand-picked context. This week the Co. app finds its own context: it cuts Larkspur's 61 documents into small pieces (**chunks**), retrieves the most relevant chunks for a question, and answers from them.

This design is called **retrieval-augmented generation (RAG)**: first retrieve relevant text, then generate the answer from it. You'll build each piece of a retriever (the search part) yourself, measure which one finds the right documents, then plug the best one into **Co. app v2** and evaluate it end to end. That includes the trap questions where an old document version is waiting to mislead it.

**Before you start**
- Connect to the KU VPN and run `docker compose up`.
- Open this notebook from `labs/lab08-rag/`, so `import coapp` finds the `coapp/` folder next to it.
- The first embedding pass over the corpus takes a few minutes. The results are cached in `.coapp_cache/`, so it only happens once.

## Words you'll meet in this lab

You don't need to memorize these. Each one is explained again where it first matters; come back here if a word stops you.

| Term | What it means | Why it matters |
|---|---|---|
| **RAG (retrieval-augmented generation)** | An app that first **retrieves** (searches for) relevant text, then puts it in the model's context so the model **generates** its answer from it. | It's how an assistant answers from a company's own documents. |
| **Retriever** | The search part of a RAG app: given a question, it returns the best-matching pieces of text. | If it misses the right document, the model can't answer correctly. |
| **Chunk, chunking** | A chunk is a small piece of a document (here up to about 1,000-1,500 characters) plus its metadata. Chunking is how you cut documents into chunks. | Retrievers return chunks, not whole documents, so the cut decides what the model sees (A1). |
| **Metadata** | Facts about a chunk's document: `doc_id`, `version`, `effective_date`, `title`, `section` (from Lab 07). | Lets the app cite sources and tell old versions from current ones. |
| **Index** | Data prepared ahead of time so searching is fast, like the index at the back of a book. | You build one for BM25 and one of embeddings. |
| **Term-based retrieval, BM25** | Ranking chunks by which words of the question they contain. BM25 is the standard formula: rare words count more, and long chunks don't win just by being long. | Fast, needs no model, great for exact names, IDs, and prices (A2). |
| **Tokenize, stop words** | Here, tokenizing means splitting text into lowercase words. Stop words are very common words ("the", "is") that are dropped because they say little. | It decides which words BM25 can match. |
| **Embedding, vector** | An embedding model turns a text into a **vector**: a long list of numbers that captures its meaning. Texts with similar meaning get similar vectors. | Lets retrieval match meaning even when the words differ (A3). |
| **Cosine similarity, dot product** | Cosine similarity measures how closely two vectors point the same way (1 = same direction, near 0 = unrelated). For vectors scaled to length 1 (**normalized**) it equals the dot product: multiply matching numbers and add them up. | It's the score embedding search ranks by. |
| **Hybrid retrieval, RRF** | Combining two retrievers' rankings. **Reciprocal rank fusion (RRF)** merges them using only the rank positions. | BM25 and embeddings fail on different questions, so together they often do better (A4). |
| **Reranker (cross-encoder)** | A second model that reads the question and one candidate chunk together and scores how relevant it is. | More accurate than the first search but slower, so it only reorders the top candidates (A5). |
| **Fallback** | What the code does when an optional part fails, such as keeping the original order if the reranker is down. | One broken component shouldn't take down the whole app. |
| **Top-k** | The k best results, for example the top 5 chunks. | The app only gives the model the top-k chunks. |
| **recall@k, MRR** | recall@k: the share of the right documents that appear in the top k. MRR (mean reciprocal rank): rewards putting a right result first (1 for rank 1, 1/2 for rank 2, ...). | How you measure a retriever without involving the generator (A6). |
| **Stale chunk** | A chunk from an outdated version of a document when a newer version exists. | The source of trap-question mistakes (B2, B4). |
| **Batch, rate limit, cache** | A batch is many texts sent in one request. Your key has a limit on requests and tokens per minute. A cache saves results to disk so a rerun doesn't repeat the work. | Embedding the whole corpus is the biggest job in the lab; batching and caching keep it fast and within limits. |
| **LLM-as-judge, gold set** | A model grading answers against the reference answers of the course's gold questions (from Labs 04-07). | Used for the end-to-end evaluation in B4. |

## Setup

Run this cell. Besides the course helpers it loads four libraries: **NumPy** (fast math on lists and tables of numbers, used for the embeddings), **pandas** (tables), **httpx** (sends web requests, used for the reranker), and **rank_bm25** (a ready-made BM25 implementation). `chat()` sends messages to a model and returns the reply text plus its latency. The last lines load the documents and the gold questions and pick the ones that have known source documents (`source_doc_ids`): those are the questions you can grade a retriever on.

In [ ]:
import json, time, re, hashlib, importlib, os
from pathlib import Path

import numpy as np
import pandas as pd
import httpx
from rank_bm25 import BM25Okapi

from itec632 import get_client, MODELS, NO_THINK, strip_think, settings
from itec632.grading import LabChecker
from itec632.corpus import corpus_dir, load_docs

client = get_client()
ck = LabChecker("lab08", part="cluster")
BIG = MODELS["main"]


def chat(messages, model=BIG, **kwargs):
    """Send chat messages. Returns (text, info). Thinking off, temperature 0."""
    kwargs.setdefault("temperature", 0)
    kwargs.setdefault("max_tokens", 500)
    t0 = time.perf_counter()
    r = client.chat.completions.create(model=model, messages=messages, extra_body=NO_THINK, **kwargs)
    return strip_think(r.choices[0].message.content or ""), {"latency_s": round(time.perf_counter() - t0, 3)}


DOCS = load_docs()
EVAL = corpus_dir() / "eval"
QA = {q["qid"]: q for q in map(json.loads, open(EVAL / "gold_qa.jsonl"))}
# Retrieval evaluation set: every gold question that has source documents (data and unanswerable questions have none).
RETR_QIDS = [q for q, v in QA.items() if v["source_doc_ids"]]
print(len(DOCS), "documents;", len(QA), "gold questions;", len(RETR_QIDS), "with source documents")

text, _ = chat([{"role": "user", "content": "Reply with just the word OK."}], max_tokens=5)
print(repr(text))
ck.check("gateway_reachable", len(text) > 0, points=1,
         hint="No reply. Check the VPN and your .env, then run notebooks/00_setup_check.ipynb.");

# Part A - Guided practice: build and measure a retriever

## A1. Chunking: fixed size versus Markdown headings

Retrievers return **chunks**, not whole documents. Chapter 6 ("Chunking strategy") lays out the trade-offs:
- Small chunks are precise, and more of them fit in the context. But they lose surrounding information, and there are more of them to embed and search.
- Large chunks keep related information together, but fewer of them fit in the model's context.
- Fixed-size chunks are simple, but they can cut sentences, tables, and steps in half (overlap reduces this).
- Splitting on the document's own structure (sections first, then smaller units only if a section is too long; the chapter calls this recursive splitting) keeps related text together.

Below, `chunk_fixed()` is done for you. Every chunk carries the document's metadata, so you can cite it and tell versions apart later.

In [ ]:
def make_chunk(doc, text, section=""):
    return {"doc_id": doc.get("doc_id", ""), "version": str(doc.get("version", "")), "title": doc.get("title", ""),
            "effective_date": str(doc.get("effective_date", "")), "section": section, "text": text}


def chunk_fixed(doc, size=1000, overlap=200):
    """Fixed-size character windows with overlap."""
    text, out, start = doc["text"], [], 0
    while start < len(text):
        out.append(make_chunk(doc, text[start:start + size]))
        if start + size >= len(text):
            break
        start += size - overlap
    return out


def split_long(text, max_chars):
    """Split an oversized section into pieces of at most max_chars, at paragraph breaks where possible."""
    if len(text) <= max_chars:
        return [text]
    pieces, buf = [], ""
    for para in text.split("\n\n"):
        while len(para) > max_chars:
            if buf:
                pieces.append(buf); buf = ""
            pieces.append(para[:max_chars]); para = para[max_chars:]
        if buf and len(buf) + 2 + len(para) > max_chars:
            pieces.append(buf); buf = para
        else:
            buf = f"{buf}\n\n{para}" if buf else para
    if buf:
        pieces.append(buf)
    return pieces


def merge_small(sections, min_chars, max_chars):
    """Glue a tiny section (for example a lone heading) onto the next one."""
    merged, buf = [], ""
    for s in sections:
        if buf and len(buf) < min_chars and len(buf) + len(s) + 2 <= max_chars:
            buf = f"{buf}\n\n{s}"
        else:
            if buf:
                merged.append(buf)
            buf = s
    if buf:
        merged.append(buf)
    return merged


def heading_of(section):
    return section.splitlines()[0].lstrip("# ").strip() if section.startswith("#") else ""

**Your turn.** Finish `chunk_by_heading()`. Split the document text at Markdown headings. The regex (a text pattern; Lab 07 used one for phone numbers) `re.split(r"\n(?=#{1,4} )", "\n" + text)` splits *before* each line that starts with 1-4 `#` characters and keeps the heading with its section. (A Markdown heading is a line starting with `#`, `##`, and so on; Larkspur's documents use them for section titles.) Merging tiny sections is done for you. Then, for each section, cut it with `split_long()` and make a chunk with `make_chunk(doc, piece, section=heading_of(sec))`.

In [ ]:
def chunk_by_heading(doc, max_chars=1500, min_chars=250):
    # TODO 1: split doc["text"] into sections at Markdown headings (see the regex above).
    sections = []
    sections = merge_small([s.strip() for s in sections if s.strip()], min_chars, max_chars)
    chunks = []
    for sec in sections:
        # TODO 2: break long sections into pieces with split_long(), and turn each piece into a chunk with make_chunk() (heading_of() gives the section name).
        pass
    return chunks


CHUNKS = {"fixed": [c for d in DOCS for c in chunk_fixed(d)],
          "heading": [c for d in DOCS for c in chunk_by_heading(d)]}
for name, ch in CHUNKS.items():
    lens = [len(c["text"]) for c in ch] or [0]
    print(f"{name:8} {len(ch):5} chunks   median {int(np.median(lens))} chars   max {max(lens)}")

# What a fixed-size cut looks like: the start of a chunk in the middle of a runbook.
print("\n--- a fixed-size chunk starts like this ---\n", CHUNKS["fixed"][101]["text"][:300])
if CHUNKS["heading"]:
    print("\n--- a heading chunk starts like this ---\n", CHUNKS["heading"][101]["text"][:300])

**Reading the output:** one line per chunking method: how many chunks it produced from all 61 documents, the median chunk length (half the chunks are shorter), and the longest chunk, all in characters. Below that you see how one chunk of each kind begins. A fixed-size chunk often starts in the middle of a sentence; a heading chunk starts at a section title.

In [ ]:
hc = CHUNKS["heading"]
ck.check("a1_heading_chunks_count", 300 <= len(hc) <= 1500, points=1,
         hint="Expected several hundred heading chunks. Did you split with the regex and loop over split_long() pieces?")
ck.check("a1_heading_chunk_size", bool(hc) and max(len(c["text"]) for c in hc) <= 1500, points=1,
         hint="No chunk should exceed max_chars; use split_long(sec, max_chars).")
ck.check("a1_heading_starts", bool(hc) and np.mean([c["text"].startswith("#") for c in hc]) >= 0.7, points=1,
         hint="Most heading chunks should start with a '#' heading line; keep the heading with its section.")
ck.record("a1_n_fixed", len(CHUNKS["fixed"])); ck.record("a1_n_heading", len(hc));

## A2. Term-based retrieval with BM25

Chapter 6 calls this **term-based retrieval**. BM25 scores a chunk by how many query terms it contains, weighting rare terms more (TF-IDF style: term frequency, how often a word appears in the chunk, combined with inverse document frequency, how rare the word is across all chunks) and normalizing by length, so long chunks don't win just by containing more words. It's fast, needs no model, and is very good at exact terms: product names, IDs, prices. It knows nothing about meaning, though, so "phone" and "call" are unrelated words to it.

**Your turn.** Build a BM25 **index** over a chunk list (the word statistics for every chunk, computed once so each search is fast) and write the search function. `tokenize()` is given: it lowercases the text, splits it into words, and drops **stop words** (very common words in `STOP`, such as "the" and "is", that say little about the topic).

The demo below prints the top 5 chunks for a question, best first, as `doc_id`, version, and section heading.

In [ ]:
STOP = set("a an the is are was were be been to of in on for and or what how do does did i we our it its this "
           "that with at by from as can my your who when which there their should would will if not any".split())


def tokenize(text):
    return [t for t in re.findall(r"[a-z0-9]+(?:\.[0-9]+)?", (text or "").lower()) if t not in STOP]


def make_bm25(chunks):
    # TODO: return BM25Okapi over the tokenized "title + section + text" of each chunk.
    return None


def bm25_search(bm25, query, k=10):
    """Return the indexes of the top-k chunks, best first."""
    # TODO: score every chunk against the tokenized query (bm25.get_scores does this), then return the indexes of the k highest scores.
    return []


BM25 = {name: make_bm25(ch) for name, ch in CHUNKS.items() if ch}
demo_q = "What number does PagerLine use for after-hours calls?"
top = bm25_search(BM25["heading"], demo_q, k=5) if BM25.get("heading") is not None else []
for i in top:
    c = CHUNKS["heading"][i]
    print(c["doc_id"], "v" + c["version"], "|", c["section"][:60])

In [ ]:
ck.check("a2_bm25_search", len(top) == 5 and any(CHUNKS["heading"][i]["doc_id"] == "RB-GEN-002" for i in top), points=2,
         hint="bm25_search should return 5 chunk indexes; for the PagerLine question an RB-GEN-002 chunk should be in them.");

## A3. Embedding retrieval

**Embedding-based retrieval** (Chapter 6) turns each chunk and the query into **vectors** (long lists of numbers that capture meaning) with an embedding model, the cluster's `embed` alias, then ranks chunks by **cosine similarity**: how closely two vectors point in the same direction (1 means the same direction, near 0 means unrelated). It can match meaning across different words. The costs: every chunk must be embedded once, and each query needs an embedding call.

`embed_texts()` sends **batches** of texts per request (32 at a time) and pauses between batches to respect your key's limit of tokens per minute. `embed_corpus()` caches the vectors in a `.npy` file (NumPy's file format for saving arrays of numbers), keyed by a fingerprint of the exact texts, so rerunning costs nothing. The vectors come back **normalized** (scaled to length 1), so cosine similarity is just a **dot product**: multiply the matching numbers of the two vectors and add them up.

The last line prints the **shape** of each embedding array: (number of chunks, numbers per vector). Each row is one chunk's vector.

In [ ]:
CACHE_DIR = Path(".coapp_cache")      # the Co. app uses the same cache, so Part B can reuse these vectors
EMBED_BATCH, TOKENS_PER_MIN = 32, 150_000


def embed_texts(texts, batch_size=EMBED_BATCH):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = [t[:6000] for t in texts[i:i + batch_size]]
        t0 = time.time()
        resp = client.embeddings.create(model=MODELS["embed"], input=batch)
        out.extend(d.embedding for d in sorted(resp.data, key=lambda d: d.index))
        if i + batch_size < len(texts):        # pace: stay near half the 300k tokens/minute key limit
            time.sleep(max(0.0, sum(map(len, batch)) / 4 / TOKENS_PER_MIN * 60 - (time.time() - t0)))
    arr = np.array(out, dtype=np.float32)
    return arr / (np.linalg.norm(arr, axis=1, keepdims=True) + 1e-9)


def embed_text_of(c):
    return f"{c['title']} - {c.get('section', '')}\n{c['text']}"


def embed_corpus(chunks):
    texts = [embed_text_of(c) for c in chunks]
    h = hashlib.sha256(MODELS["embed"].encode())
    for t in texts:
        h.update(t.encode())
    path = CACHE_DIR / f"emb_{h.hexdigest()[:16]}.npy"
    if path.exists():
        print(f"  loaded {path}")
        return np.load(path)
    t0 = time.time()
    E = embed_texts(texts)
    CACHE_DIR.mkdir(exist_ok=True)
    np.save(path, E)
    print(f"  embedded {len(texts)} chunks in {time.time() - t0:.0f}s with {-(-len(texts) // EMBED_BATCH)} requests -> {path}")
    return E


EMB = {}
for name, ch in CHUNKS.items():
    if ch:
        print(name)
        EMB[name] = embed_corpus(ch)
QVEC = embed_texts([QA[q]["question"] for q in RETR_QIDS])     # all eval questions in a few batched calls
QVEC = dict(zip(RETR_QIDS, QVEC))
print({k: v.shape for k, v in EMB.items()})

**Your turn.** Write `embed_search()`: cosine similarity between the query vector and every chunk vector (the rows of `E`), then the top-k indexes. Because the rows are normalized, `E @ qvec` gives all the similarities at once (`@` is Python's matrix multiplication: it takes the dot product of `qvec` with every row of `E`). `np.argsort(-sims)` then lists the row numbers from highest to lowest similarity.

In [ ]:
def embed_search(E, qvec, k=10):
    """Indexes of the k chunks most similar to qvec, best first."""
    # TODO: compute every chunk's similarity to qvec (the embeddings are normalized, so one matrix-vector product does it), then return the indexes of the k most similar chunks.
    return []


qv = embed_texts(["Who do I phone when the on-call engineer doesn't pick up?"])[0]
top_e = embed_search(EMB["heading"], qv, k=5) if "heading" in EMB else []
for i in top_e:
    c = CHUNKS["heading"][i]
    print(c["doc_id"], "v" + c["version"], "|", c["section"][:60])

In [ ]:
ck.check("a3_embeddings_shape", "heading" in EMB and EMB["heading"].shape[0] == len(CHUNKS["heading"]), points=1,
         hint="Each heading chunk needs one embedding row. Rerun A1 and the embedding cell.")
ck.check("a3_embed_search", len(top_e) == 5 and all(isinstance(i, int) for i in top_e) and len(set(top_e)) == 5, points=1,
         hint="embed_search should return 5 distinct int indexes (convert numpy ints with int(i)).");

## A4. Hybrid retrieval with reciprocal rank fusion

BM25 and embeddings fail on different questions, so production systems often combine them. Chapter 6 calls this "Combining retrieval algorithms". **Reciprocal rank fusion (RRF)** merges rankings without having to compare their incompatible scores (a BM25 score and a cosine similarity are on different scales). It gives each item `1 / (k + rank)` from each ranking it appears in (rank starts at 1, k is usually 60), and the totals decide the order.

**Your turn.** Implement `rrf()`.

In [ ]:
def rrf(rankings, k=60):
    """rankings: list of lists of ids (best first). Returns the list of ids sorted by fused score, best first."""
    scores = {}
    # TODO: for every ranking, add 1 / (k + rank) to each item's score, where the best item in a ranking has rank 1.
    return sorted(scores, key=scores.get, reverse=True)


print(rrf([[1, 2, 3], [3, 1, 4]]))

In [ ]:
ck.check("a4_rrf_unit", rrf([[1, 2, 3], [3, 1, 4]]) == [1, 3, 2, 4] and rrf([["a"], ["b"], ["b"]])[0] == "b", points=2,
         hint="Expected rrf([[1,2,3],[3,1,4]]) == [1, 3, 2, 4]. Each ranking adds 1/(k+rank) with rank starting at 1.");

## A5. Reranking

Chapter 6 ("Reranking", and "Combining retrieval algorithms") describes the pattern: a cheap retriever fetches candidates, then a more precise but more expensive mechanism reorders them. The cluster's reranker (Qwen3-Reranker) is a **cross-encoder**: it reads the query and one candidate *together* and scores their relevance. That's more accurate than comparing two separately computed embeddings, but slower, so you rerank only the top 20 or so candidates from a cheaper retriever.

The gateway serves `rerank` with a Cohere-style API (the same request format as Cohere's rerank service): `POST {gateway without /v1}/rerank` with `{"model", "query", "documents", "top_n"}`. `rerank()` below makes that call: an HTTP POST request that sends the query and the candidate texts to that web address and gets a score for each text back. If the call fails (server down, model not loaded), it returns `None` instead of crashing. Your code must then **fall back** to the original order and record the failure. Never let an optional component take the whole app down.

**Reading the output:** the demo prints a list of `(index, score)` pairs, best first. `index` is the position of the text in the list you sent (0 is the first) and `score` is its relevance. `RERANK_LOG` counts successful and failed calls and keeps the last error. If the reranker is down you'll see `None` and a failed count.

In [ ]:
BASE_URL, API_KEY = settings()
RERANK_URL = (BASE_URL.rstrip("/")[:-3] if BASE_URL.rstrip("/").endswith("/v1") else BASE_URL.rstrip("/")) + "/rerank"
RERANK_LOG = {"ok": 0, "failed": 0, "last_error": None}


def rerank(query, texts, top_n=None):
    """Return [(index_into_texts, score)] best first, or None if the rerank call fails."""
    try:
        r = httpx.post(RERANK_URL, headers={"Authorization": f"Bearer {API_KEY}"}, timeout=60,
                       json={"model": MODELS["rerank"], "query": query, "documents": texts, "top_n": top_n or len(texts)})
        r.raise_for_status()
        RERANK_LOG["ok"] += 1
        return [(int(x["index"]), float(x["relevance_score"])) for x in r.json()["results"]]
    except Exception as e:
        RERANK_LOG["failed"] += 1
        RERANK_LOG["last_error"] = f"{type(e).__name__}: {str(e)[:150]}"
        return None


print(rerank("after-hours phone number", ["Call PagerLine at (913) 555-0142.", "The office has a coffee machine."]))
print(RERANK_LOG)

**Your turn.** Write `rerank_ids()`. Send the candidate chunks' texts (use `embed_text_of(chunk)`) to `rerank()`. If it returns `None`, return `cand_ids` unchanged (the fallback). Otherwise return the candidate ids in the reranker's order. The demo prints the chunk ids before and after reranking: the same numbers, possibly in a new order.

In [ ]:
def rerank_ids(query, cand_ids, chunks):
    """Reorder candidate chunk ids with the reranker; fall back to the input order if reranking fails."""
    # TODO: call rerank() on the candidates' texts; on None return cand_ids; else map result indexes back to ids.
    return cand_ids


demo = bm25_search(BM25["heading"], demo_q, k=10) if BM25.get("heading") is not None else []
print("before:", demo)
print("after: ", rerank_ids(demo_q, demo, CHUNKS["heading"]))

In [ ]:
after = rerank_ids(demo_q, demo, CHUNKS["heading"]) if demo else []
ck.check("a5_rerank_permutation", bool(demo) and sorted(after) == sorted(demo), points=1,
         hint="rerank_ids must return the same ids in a new order (or unchanged if reranking failed).")
saved = RERANK_LOG.copy(); RERANK_URL_SAVED = RERANK_URL
RERANK_URL = "http://127.0.0.1:9/rerank"                     # simulate an outage
fallback = rerank_ids(demo_q, demo, CHUNKS["heading"]) if demo else []
RERANK_URL = RERANK_URL_SAVED
ck.check("a5_rerank_fallback", bool(demo) and fallback == demo, points=1,
         hint="When rerank() returns None, rerank_ids must return cand_ids unchanged.")
RERANK_LOG.update(saved)

## A6. Evaluate retrieval: recall@k and MRR

Chapter 6 ("Comparing retrieval algorithms") evaluates retrievers with **context precision** and **context recall**, and points to ranking metrics such as **MRR** and NDCG (another ranking metric, not used here) when order matters. With the gold set's `source_doc_ids` (the documents a person marked as containing each answer), we measure at the **document** level:
- **recall@k** is the fraction of a question's gold documents that appear among the doc_ids of the top-k chunks. If the gold is `["PRICE-2026", "SVC-CAT-2026"]` and only PRICE-2026 is retrieved, recall is 0.5.
- **MRR** (mean reciprocal rank) is 1 / (rank of the first chunk whose doc_id is a gold document), or 0 if none in the top k. It rewards putting a right chunk first.

**Your turn.** Implement both functions for one question. Averaging over questions is done for you.

In [ ]:
def recall_at_k(retrieved_doc_ids, gold_doc_ids, k):
    """Fraction of gold doc_ids present in the first k retrieved doc_ids."""
    # TODO
    return 0.0


def reciprocal_rank(retrieved_doc_ids, gold_doc_ids, k=10):
    """1/rank of the first retrieved doc_id that is in gold (rank starts at 1), else 0."""
    # TODO
    return 0.0


print(recall_at_k(["A", "B", "A", "C"], ["A", "C"], k=3), reciprocal_rank(["B", "A"], ["A"]))

In [ ]:
ck.check("a6_metric_unit", recall_at_k(["A", "B", "A", "C"], ["A", "C"], 3) == 0.5 and recall_at_k(["A", "C"], ["A", "C"], 2) == 1.0
         and reciprocal_rank(["B", "A"], ["A"]) == 0.5 and reciprocal_rank(["B"], ["A"]) == 0.0, points=2,
         hint="recall_at_k(['A','B','A','C'], ['A','C'], 3) should be 0.5; reciprocal_rank(['B','A'], ['A']) should be 0.5.");

Now compare everything on the 65 gold questions that have source documents. Each method returns chunk indexes, and we score their doc_ids. Reranking costs one request per question (about 65), so it runs only on heading chunks with hybrid candidates. This cell takes a minute or two.

In [ ]:
def method_rankings(name, qid, depth=20):
    q, ch = QA[qid]["question"], CHUNKS[name]
    bm = bm25_search(BM25[name], q, k=depth)
    em = embed_search(EMB[name], QVEC[qid], k=depth)
    return {"bm25": bm, "embedding": em, "hybrid": rrf([bm, em])[:depth]}


rows, hybrid_heading = [], {}
for name in CHUNKS:
    if not CHUNKS[name] or BM25.get(name) is None:
        continue
    for qid in RETR_QIDS:
        gold = QA[qid]["source_doc_ids"]
        for method, ids in method_rankings(name, qid).items():
            docs = [CHUNKS[name][i]["doc_id"] for i in ids]
            rows.append({"chunking": name, "method": method, "qid": qid, "recall@5": recall_at_k(docs, gold, 5),
                         "recall@10": recall_at_k(docs, gold, 10), "mrr": reciprocal_rank(docs, gold)})
            if name == "heading" and method == "hybrid":
                hybrid_heading[qid] = ids

for qid in RETR_QIDS:                                   # rerank the hybrid heading candidates (1 request each)
    if qid not in hybrid_heading:
        continue
    ids = rerank_ids(QA[qid]["question"], hybrid_heading[qid], CHUNKS["heading"])
    docs = [CHUNKS["heading"][i]["doc_id"] for i in ids]
    gold = QA[qid]["source_doc_ids"]
    rows.append({"chunking": "heading", "method": "hybrid+rerank", "qid": qid, "recall@5": recall_at_k(docs, gold, 5),
                 "recall@10": recall_at_k(docs, gold, 10), "mrr": reciprocal_rank(docs, gold)})
    time.sleep(0.3)

retr = pd.DataFrame(rows)
retr_table = retr.groupby(["chunking", "method"])[["recall@5", "recall@10", "mrr"]].mean().round(3) if len(retr) else pd.DataFrame()
print(retr_table)
print("rerank calls:", RERANK_LOG)

**Reading the table:** one row per combination of chunking (`fixed`, `heading`) and retrieval method (`bm25`, `embedding`, `hybrid`, and `hybrid+rerank` for heading chunks only). Each number is an average over the 65 questions: `recall@5` and `recall@10` are the share of gold documents found in the top 5 and top 10, and `mrr` is how close to the top the first right chunk was (1.0 = always first). Higher is better in every column. `rerank calls` shows how many rerank requests succeeded and failed; if any failed, those questions kept the hybrid order.

In [ ]:
ck.check("a6_table_complete", len(retr_table) >= 7, points=2,
         hint="The table needs 3 methods x 2 chunkings + hybrid+rerank. Finish A1-A5 so every method returns results.")
best = float(retr_table["recall@5"].max()) if len(retr_table) else 0.0
ck.check("a6_best_recall5", best >= 0.6, points=2,
         hint="Best recall@5 is below 0.6. Check chunk_by_heading and that searches return chunk indexes for the right chunk list.")
for (name, method), r in retr_table.iterrows():
    ck.record(f"a6_{name}_{method}_recall5", float(r["recall@5"]))
    ck.record(f"a6_{name}_{method}_mrr", float(r["mrr"]))
ck.record("a6_rerank_failed_calls", RERANK_LOG["failed"])
if RERANK_LOG["failed"]:
    print("Reranking failed", RERANK_LOG["failed"], "times; hybrid+rerank fell back to hybrid order. Last error:", RERANK_LOG["last_error"])

Which questions does each method get wrong? Seeing the failures helps more than the averages. The cell lists the questions where BM25 and embeddings disagree most, at the document level, top 5, on heading chunks.

**Reading the output:** each line is one question. `bm25-embedding` is BM25's recall@5 minus the embedding recall@5 for that question: a positive number means BM25 found more of the gold documents, a negative number means embeddings did. Then come the gold documents and the start of the question. The first four lines favor embeddings, the last four favor BM25.

In [ ]:
if len(retr):
    piv = retr[retr.chunking == "heading"].pivot_table(index="qid", columns="method", values="recall@5")
    diff = (piv["bm25"] - piv["embedding"]).sort_values()
    show = pd.concat([diff.head(4), diff.tail(4)])
    for qid, d in show.items():
        print(f"{qid}  bm25-embedding = {d:+.2f}  gold={QA[qid]['source_doc_ids']}  {QA[qid]['question'][:90]}")

# Part B - Applied lab: Co. app v2

The `coapp/` folder next to this notebook is version 2 of the company assistant. It has the same interface as Lab 7, plus `retrieve.py`:

```
coapp/
  config.py      # MODEL, TOP_K = 5, CHUNK_CHARS, USE_RERANK, PREFER_CURRENT, cache dir   (complete)
  prompts.py     # the Lab 7 reference prompt (you may swap in your own Lab 7 prompts.py)
  retrieve.py    # build_index(docs) -> index; retrieve(index, question, k=5) -> list[Chunk]   <- YOU (3 TODOs)
  assistant.py   # answer(question, history=None) -> dict, now with retrieval            <- YOU (1 TODO)
```

`retrieve()` does hybrid BM25 + embedding retrieval with RRF, then **prefers current versions**: if several versions of one doc_id come back, only the newest stays. It can also rerank (off by default; set the **environment variable** `COAPP_USE_RERANK=1`, a setting the app reads from the system when it starts, or `config.USE_RERANK = True` if your A6 results say it helps).

`coapp_reference/` is a complete v2. Lab 9 starts from it.

Edit the files in the JupyterLab editor, save, and rerun the reload cell. (As in Lab 07, Python reads an imported file only once; the reload cell makes it read your saved changes and also throws away the old index.)

In [ ]:
import coapp, coapp.config, coapp.prompts, coapp.retrieve, coapp.assistant


def reload_coapp():
    for m in (coapp.config, coapp.prompts, coapp.retrieve, coapp.assistant):
        importlib.reload(m)
    coapp.assistant._INDEX = None
    print("coapp reloaded")


reload_coapp()

## B1-B2. `retrieve.py`

Open `coapp/retrieve.py`:
- **B1:** paste your Part A logic into `chunk_doc()` (same as `chunk_by_heading`) and `rrf_fuse()`. Note that `rrf_fuse` returns `(id, score)` pairs.
- **B2:** write `prefer_current()`. For each doc_id, keep only the chunks with the newest `effective_date` among the retrieved ones. Older ones are **stale**: a newer version of the same document exists. (Chapter 6, "Reranking", notes that results can also be reranked by time, giving more weight to recent data; this is a strict version of that idea.)

Save the file and run the checks.

In [ ]:
reload_coapp()
R = coapp.retrieve
doc = next(d for d in DOCS if d["doc_id"] == "RB-PRT-001" and d["version"] == "2")
b_chunks = R.chunk_doc(doc)
print(len(b_chunks), "chunks from RB-PRT-001 v2; first:", b_chunks[0]["section"] if b_chunks else None)
fused = R.rrf_fuse([[1, 2, 3], [3, 1, 4]])
print(fused)
toy = [{"doc_id": "RB-X", "effective_date": "2023-01-01", "v": "old"}, {"doc_id": "RB-X", "effective_date": "2025-01-01", "v": "new"},
       {"doc_id": "PRICE", "effective_date": "2026-01-01", "v": "p"}, {"doc_id": "RB-X", "effective_date": "2023-01-01", "v": "old2"}]
kept, dropped = R.prefer_current(toy)
print([c["v"] for c in kept], dropped)

In [ ]:
keys = {"doc_id", "version", "title", "effective_date", "section", "text"}
ck.check("b1_chunk_doc", len(b_chunks) >= 3 and all(keys <= set(c) for c in b_chunks) and max(len(c["text"]) for c in b_chunks) <= coapp.config.CHUNK_CHARS,
         points=2, hint="coapp.retrieve.chunk_doc should split by headings like your Part A function and keep all metadata keys.")
ck.check("b1_rrf_fuse", [i for i, _ in fused] == [1, 3, 2, 4] and abs(fused[0][1] - (1 / 61 + 1 / 62)) < 1e-9, points=1,
         hint="rrf_fuse returns [(id, score)] sorted best first, with score = sum of 1/(k + rank).")
ck.check("b2_prefer_current", [c["v"] for c in kept] == ["new", "p"] and dropped == 2, points=2,
         hint="Keep only chunks whose effective_date is the newest for their doc_id; keep the order; return (kept, n_dropped).");

## B3. Plug retrieval into `answer()`

Open `coapp/assistant.py` and complete the TODO in `answer()`: call `retrieve(...)`. Save the file and run the **smoke test** (a quick run that shows whether the basics work at all). The first call builds the index. If your chunking matches Part A, the embeddings load from `.coapp_cache/`. Otherwise the corpus is embedded again, which takes a few minutes.

In [ ]:
reload_coapp()
t0 = time.time()
index = coapp.assistant.get_index()
print(f"index: {len(index['chunks'])} chunks, embeddings {None if index['emb'] is None else index['emb'].shape}, "
      f"error={index['embed_error']}  ({time.time() - t0:.0f}s)")
smoke = coapp.assistant.answer("What is the hourly rate for out-of-scope work after hours?")
print(smoke["answer"], "\n")
print("sources:", smoke["sources"])
print("retrieved:", smoke["retrieved"])
print("retrieval:", smoke["retrieval"])

**Reading the output:** the `index` line shows how many chunks were indexed, the shape of the embedding array (chunks, numbers per vector), and any embedding error (`None` is good). Then come the answer and three lists. `sources` are the documents the answer cites (if it cites none, all retrieved documents). `retrieved` are the top-k chunks retrieval handed the model, with their fused score. `retrieval` describes what the retriever did: which retrievers ran, whether reranking was used or failed, and how many stale chunks were dropped.

In [ ]:
spec_keys = {"answer", "sources", "tool_calls", "model", "latency_s"}
ck.check("b3_answer_spec", spec_keys <= set(smoke) and smoke["tool_calls"] == [] and all({"doc_id", "version", "title"} <= set(s) for s in smoke["sources"]),
         points=1, hint="answer() must return the course dict: answer, sources [{doc_id, version, title}], tool_calls, model, latency_s.")
ck.check("b3_retrieval_used", len(smoke.get("retrieved", [])) == coapp.config.TOP_K, points=2,
         hint="answer() retrieved nothing. Complete TODO (B3) in coapp/assistant.py: call retrieve() with the index, the question, the right k, and retrieve_kwargs.");

## B4. End-to-end evaluation

Now evaluate the whole app on 30 gold questions it has never seen with hand-picked context. Retrieval decides the context this time:
- **12 trap questions:** an outdated document exists somewhere in the corpus;
- **12 regular questions;**
- **6 unanswerable questions:** the right behavior is to say "I don't know".

Data questions are left out. Document retrieval can't count tickets, and Lab 9 adds SQL tools for that.

For each question you record:
- retrieval recall@5 against the gold doc_ids;
- the judge's verdict;
- whether any retrieved chunk is **stale** (a newer version of its doc_id exists).

Trap questions also run a second time with `prefer_current_version=False`, so you can measure what the metadata filter buys you. A second, cheap grader, `keyword_match()`, checks whether every number in the reference answer also appears in the app's answer, so you can see how often it agrees with the judge. About 115 requests. Results are cached in `lab08_app_results.jsonl`.

In [ ]:
B_QIDS = ["Q021", "Q014", "Q029", "Q032", "Q047", "Q084", "Q076", "Q002", "Q077", "Q011", "Q039", "Q022",   # traps
          "Q003", "Q007", "Q012", "Q015", "Q016", "Q027", "Q037", "Q044", "Q050", "Q054", "Q064", "Q072",   # regular
          "Q001", "Q005", "Q009", "Q052", "Q059", "Q082"]                                                   # unanswerable

JUDGE_FORMAT = {"type": "json_schema", "json_schema": {"name": "verdict", "strict": True, "schema": {
    "type": "object", "properties": {"verdict": {"type": "string", "enum": ["correct", "incorrect", "abstained"]},
                                     "reason": {"type": "string"}},
    "required": ["verdict", "reason"], "additionalProperties": False}}}
JUDGE_SYSTEM = """You grade answers from Larkspur Ridge IT's help-desk assistant against a reference answer
written by an expert who read the current documents. Choose one verdict:
- "correct": the answer states the key facts of the reference (numbers, names, steps) and does not contradict it.
  Different wording or extra correct detail is fine.
- "incorrect": the answer contradicts the reference, gives a different number, name, or procedure, or misses the key fact.
- "abstained": the answer says it does not know or that the documents do not contain the answer.
Judge only against the reference, not your own knowledge. Give a one-sentence reason."""


def judge(qid, answer_text):
    q = QA[qid]
    out, _ = chat([{"role": "system", "content": JUDGE_SYSTEM},
                   {"role": "user", "content": f"Question: {q['question']}\n\nReference answer: {q['reference_answer']}\n\n"
                                               f"Assistant answer: {answer_text}"}],
                  response_format=JUDGE_FORMAT, max_tokens=150)
    try:
        return json.loads(out)["verdict"]
    except Exception:
        return "incorrect"


def group_of(qid):
    if QA[qid]["reference_answer"].startswith("Not in the documents"):
        return "unanswerable"
    return "trap" if QA[qid]["trap"] else "regular"


def is_right(qid, verdict):
    return verdict == "abstained" if group_of(qid) == "unanswerable" else verdict == "correct"


FACT = re.compile(r"\$?\d[\d,]*(?:\.\d+)?")


def keyword_match(answer_text, reference):
    """Cheap cross-check: are all numbers in the reference also in the answer? None if the reference has no numbers."""
    facts = {f.strip("$").rstrip(".,").replace(",", "") for f in FACT.findall(reference)}
    facts = {f for f in facts if f}
    if not facts:
        return None
    ans = answer_text.replace(",", "")
    return all(f in ans for f in facts)


APP_CACHE = "lab08_app_results.jsonl"


def run_eval(qids, setting="prefer_current", **kw):
    idx = coapp.assistant.get_index()
    run_id = hashlib.sha256(json.dumps([coapp.prompts.PROMPT_VERSION, len(idx["chunks"]), coapp.config.USE_RERANK,
                                        coapp.config.TOP_K, setting]).encode()).hexdigest()[:8]
    cache = {}
    if os.path.exists(APP_CACHE):
        for line in open(APP_CACHE):
            r = json.loads(line); cache[(r["qid"], r["run_id"])] = r
    rows = []
    for qid in qids:
        if (qid, run_id) in cache:
            rows.append(cache[(qid, run_id)]); continue
        out = coapp.assistant.answer(QA[qid]["question"], **kw)
        verdict = judge(qid, out["answer"])
        docs = [c["doc_id"] for c in out["retrieved"]]
        gold = QA[qid]["source_doc_ids"]
        stale = [c for c in out["retrieved"] if c["effective_date"] < idx["latest"].get(c["doc_id"], "")]
        r = {"qid": qid, "run_id": run_id, "setting": setting, "group": group_of(qid), "verdict": verdict,
             "right": is_right(qid, verdict), "recall@5": recall_at_k(docs, gold, 5) if gold else None,
             "stale_chunks": len(stale), "retrieved": [f"{c['doc_id']} v{c['version']}" for c in out["retrieved"]],
             "keyword_match": keyword_match(out["answer"], QA[qid]["reference_answer"]) if gold else None,
             "rerank_error": out["retrieval"].get("rerank_error"), "latency_s": out["latency_s"], "answer": out["answer"]}
        rows.append(r)
        with open(APP_CACHE, "a") as f:
            f.write(json.dumps(r) + "\n")
        time.sleep(0.5)
    return pd.DataFrame(rows)


res = run_eval(B_QIDS)
TRAPS = [q for q in B_QIDS if group_of(q) == "trap"]
res_noprefer = run_eval(TRAPS, setting="no_prefer", prefer_current_version=False)

summary = res.groupby("group").agg(n=("qid", "count"), accuracy=("right", "mean"), recall5=("recall@5", "mean"),
                                   stale_chunks=("stale_chunks", "mean"), latency_s=("latency_s", "mean"))
print(summary.round(3))
abstain_rate = float((res[res.group == "unanswerable"]["verdict"] == "abstained").mean())
trap_cmp = pd.DataFrame({
    "prefer_current=True": [res[res.group == "trap"]["right"].mean(), res[res.group == "trap"]["stale_chunks"].mean()],
    "prefer_current=False": [res_noprefer["right"].mean(), res_noprefer["stale_chunks"].mean()]},
    index=["trap accuracy", "stale chunks per question"])
print("\nunanswerable: abstention rate =", round(abstain_rate, 3))
print("\n", trap_cmp.round(3))
km = res.dropna(subset=["keyword_match"])
if len(km):
    agree = (km["keyword_match"].astype(bool) == km["right"]).mean()
    print(f"\njudge vs keyword-match agreement on {len(km)} questions with numbers: {agree:.2f}")

**Reading the output:** the first table has one row per question group. `n` is the number of questions, `accuracy` the share judged right (for unanswerable questions, saying "I don't know" counts as right), `recall5` the average recall@5 against the gold documents (empty for unanswerable questions, which have none), `stale_chunks` the average number of outdated chunks retrieved per question, and `latency_s` the average seconds per answer. The abstention rate is the share of unanswerable questions where the app said it didn't know. The second table compares the 12 trap questions with and without `prefer_current`. The last line is the share of questions (among those whose reference contains numbers) where the judge and the keyword check agree.

In [ ]:
ck.check("b4_evaluated", len(res) >= 28 and len(res_noprefer) >= 10, points=2,
         hint="Evaluate all 30 questions and the 12 traps without prefer_current. Rerun; cached rows are reused.")
ck.check("b4_overall_accuracy", res["right"].mean() >= 0.5, points=2,
         hint="Overall accuracy is below 0.5. Look at recall@5: is retrieval finding the gold documents? Then read wrong answers.")
ck.check("b4_abstention", abstain_rate >= 0.5, points=1,
         hint="The app answered unanswerable questions instead of saying it doesn't know. Check the 'I don't know' rule in prompts.py.")
ck.check("b4_stale_reduced", res[res.group == "trap"]["stale_chunks"].mean() <= res_noprefer["stale_chunks"].mean(), points=1,
         hint="prefer_current should never increase stale chunks. Check prefer_current() in retrieve.py.")
ck.record("b4_overall_accuracy", float(res["right"].mean()))
ck.record("b4_trap_acc_prefer", float(res[res.group == "trap"]["right"].mean()))
ck.record("b4_trap_acc_noprefer", float(res_noprefer["right"].mean()))
ck.record("b4_regular_acc", float(res[res.group == "regular"]["right"].mean()))
ck.record("b4_abstention_rate", abstain_rate)
ck.record("b4_recall5_answerable", float(res["recall@5"].dropna().mean()))
ck.record("b4_stale_prefer", float(res[res.group == "trap"]["stale_chunks"].mean()))
ck.record("b4_stale_noprefer", float(res_noprefer["stale_chunks"].mean()))
ck.record("b4_mean_latency_s", float(res["latency_s"].mean()));

## B5. Failure analysis

Pick **three wrong answers** from `res` (or from `res_noprefer` if you have fewer than three) and diagnose each one. Assign one cause:
- `"retrieval_miss"`: the gold document wasn't in the top 5 (check `recall@5` and `retrieved`);
- `"stale_context"`: an outdated document or sentence was retrieved and the model used it;
- `"generation"`: the right chunk was retrieved, but the model misread it, mixed facts, or ignored it;
- `"judge_error"`: the answer is actually fine and the judge was wrong.

Write at least 8 words of evidence for each, pointing at what you saw.

The next cell prints every wrong answer: the question id, group, and setting; its recall@5, number of stale chunks, and the judge's verdict; then the question (`Q`), the app's answer (`A`), the reference answer (`REF`), and the chunks that were retrieved, as `doc_id` and version.

In [ ]:
wrong = pd.concat([res, res_noprefer])
wrong = wrong[~wrong["right"]].drop_duplicates("qid")
for _, r in wrong.iterrows():
    print(f"[{r['qid']} {r['group']} {r['setting']}] recall@5={r['recall@5']}  stale={r['stale_chunks']}  verdict={r['verdict']}")
    print(f"  Q: {QA[r['qid']]['question']}\n  A: {r['answer'][:300]}\n  REF: {QA[r['qid']]['reference_answer']}\n  retrieved: {r['retrieved']}\n")

In [ ]:
# TODO: three entries  "Qxxx": ("retrieval_miss" | "stale_context" | "generation" | "judge_error", "evidence, 8+ words"),
failures = {
}
for k, v in failures.items():
    print(k, v)

In [ ]:
CAUSES = {"retrieval_miss", "stale_context", "generation", "judge_error"}
good = {q: v for q, v in failures.items() if q in QA and isinstance(v, tuple) and len(v) == 2 and v[0] in CAUSES and len(str(v[1]).split()) >= 8}
ck.check("b5_three_failures", len(good) >= 3, points=3,
         hint="Add 3 entries: qid -> (cause, evidence of 8+ words). Causes: retrieval_miss, stale_context, generation, judge_error.")
ck.record("b5_causes", {c: sum(v[0] == c for v in good.values()) for c in CAUSES});

# Written analysis

Answer each question in the box under it, using **your own numbers** (they're all in the final summary) and citing Chapter 6 where asked.

**Q1. Term versus embedding versus hybrid.** From your A6 table, compare BM25, embeddings, hybrid, and hybrid+rerank on recall@5 and MRR. Use the "disagreement" list to describe one question where BM25 won and one where embeddings won, and explain why from what each method can and can't see. Cite Chapter 6's comparison of term-based and embedding-based retrieval ("Comparing retrieval algorithms", Table 6-2), and what it says in "Combining retrieval algorithms" and "Reranking". Did reranking earn its extra request per query? If it failed, what did your fallback do?

*Your answer:*

**Q2. Chunking.** Compare fixed-size and heading chunks: number of chunks, recall@5, and MRR. What did a fixed-size chunk look like at its start, and why might that hurt retrieval or generation? Using Chapter 6's discussion of chunking strategy, name one trade-off you'd revisit (chunk size, overlap, or what gets embedded) and predict its effect.

*Your answer:*

**Q3. Failure analysis.** For your three B5 failures, explain the cause and the evidence. Then say what you'd change, and where in the pipeline: retriever, chunking, metadata filter, prompt, or the judge. Which part of the RAG pipeline limits your app most right now, according to your numbers?

*Your answer:*

**Q4. Stale documents and RAG evaluation.** Report trap accuracy and stale chunks per question with and without `prefer_current`, and your abstention rate on unanswerable questions. Why can't document-level recall@k tell you whether the *right version* was retrieved? Which planted conflicts can a doc_id-based version filter never fix, and what handles those instead? Connect to Chapter 6's RAG architecture, where the retriever's output is the generator's context, so retrieval quality bounds answer quality.

*Your answer:*

# AI-use reflection

**1. What did you use AI assistance for in this lab (tools, which steps)?**

*Your answer:*

**2. Where did the AI help most, and where was it wrong or unhelpful? How did you notice?**

*Your answer:*

**3. What is one thing you now understand that you could explain without AI help?**

*Your answer:*

**4. Did you ask an AI assistant to explain or write any retrieval code (BM25, cosine similarity, RRF)? How did you check that what it gave you was correct? Name a specific check or test.**

*Your answer:*

## Final check

Run this last, save, upload the .ipynb.

In [ ]:
ck.report()